# Race Simulation — The Elimination Stage

A cup (Cup of the Week) has three stages:
1. **Qualification** — 64 players selected from the full field
2. **Race** — 39 elimination rounds determine finish positions 1–64
3. **Scoring** — finish positions mapped to points

This notebook focuses on **stage 2: the race**.
Given 64 qualifiers already selected, what finish positions does the simulation produce?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tmonacodel.config import TournamentConfig
from tmonacodel.scoring import build_finish_position_lookup
from tmonacodel.race import simulate_race

sns.set_theme(style='whitegrid')

config = TournamentConfig(n_players=128, random_seed=42)
finish_pos_lookup = build_finish_position_lookup(config)

print(f"Qualifiers per race: {config.n_qualifiers}")
print(f"Elimination stages: {config.elimination_stages}")

## The finish position lookup

The race uses a single random permutation of 64 qualifiers.
The `finish_pos_lookup` array maps a permutation index to a finish position:
- index 0 → first eliminated (worst finish)
- index 63 → last standing (1st place)

Players eliminated together in a double-elimination round get consecutive unique positions
(e.g. 48 and 49) so each of the 64 positions appears exactly once in the lookup.

In [ ]:
lookup_series = pd.Series(
    finish_pos_lookup,
    index=pd.RangeIndex(len(finish_pos_lookup), name="perm_index"),
    name="finish_position",
)
print("Worst finishes (perm_index 0-4):")
print(lookup_series.head(5).to_string())
print("\nBest finishes (perm_index 60-63):")
print(lookup_series.tail(4).to_string())
print(f"\nAll 64 positions unique: {lookup_series.nunique() == config.n_qualifiers}")

## Simulating one race

Given 64 qualifiers (indexed 0-63), `simulate_race` returns their finish positions directly.
No qualification step, no scoring — just the elimination result.

In [ ]:
rng = np.random.default_rng(42)
finish_positions = simulate_race(config.n_qualifiers, finish_pos_lookup, rng)

result_df = pd.DataFrame({
    "qualifier_index": np.arange(config.n_qualifiers),
    "finish_position": finish_positions,
}).sort_values("finish_position").reset_index(drop=True)

print("Top 10 finishers:")
result_df.head(10)

## Running 10,000 races — is the distribution uniform?

Track qualifier index 0 across 10,000 races.
With purely random mechanics every qualifier is equally likely to receive any finish position,
so the histogram should be flat across positions 1-64.

In [ ]:
n_trials = 10_000
target = 0

rng = np.random.default_rng(42)
recorded = np.empty(n_trials, dtype=np.int64)

for i in range(n_trials):
    fp = simulate_race(config.n_qualifiers, finish_pos_lookup, rng)
    recorded[i] = fp[target]

fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(recorded, bins=range(1, config.n_qualifiers + 2), edgecolor='white', linewidth=0.4)
ax.set_xlabel("Finish position")
ax.set_ylabel("Count across 10,000 races")
ax.set_title("Finish position distribution for qualifier 0 — uniform random baseline")
expected = n_trials / config.n_qualifiers
ax.axhline(expected, color='red', linestyle='--', label=f'Expected ({expected:.0f})')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean finish position: {recorded.mean():.2f} (expected {(config.n_qualifiers + 1) / 2:.2f})")

## What changes next

This flat histogram is the **random baseline** for the race stage.

When skill is introduced, the `rng.permutation` in `simulate_race` (`race.py`) becomes
weighted — better players will survive elimination rounds more often, pulling their
distribution towards lower finish positions (closer to 1st).

The qualification and scoring stages are covered in separate notebooks.